In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.model_selection import KFold, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.svm import SVR

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

from sklearn.decomposition import PCA
from sklearn.linear_model import Lasso
# For classification
from sklearn.tree import DecisionTreeClassifier

# For regression
from sklearn.tree import DecisionTreeRegressor


In [2]:
df = pd.read_csv('Islamabad(reverse)_for_model_sel.csv')

In [3]:
df.columns

Index(['property_type', 'location', 'area', 'bedRoom', 'bathroom', 'balcony',
       'floorNum', 'facing', 'agePossession', 'price_in_crore', 'Study Room',
       'Store Room', 'Servant Room', 'Prayer Room', 'Pooja Room',
       'furnishing_type', 'luxury_score', 'luxury_category', 'floor_category'],
      dtype='object')

In [4]:
df.head()

,property_type,location,area,bedRoom,bathroom,balcony,floorNum,facing,agePossession,price_in_crore,Study Room,Store Room,Servant Room,Prayer Room,Pooja Room,furnishing_type,luxury_score,luxury_category,floor_category
0,flat,DHA Phase 1,793.0,2,2,1,3,North-East,Moderately Old,1.87,0,0,0,1,0,1,17,Low,Mid floor
1,flat,G-15,870.0,2,2,3,3,East,New Property,1.87,1,0,0,0,0,1,39,Medium,Mid floor
2,flat,Bahria Town,947.0,3,3,2,6,South,Under Construction,1.84,0,1,0,0,0,2,8,Low,High floor
3,flat,G-12,1322.0,3,4,3,9,North,Old Property,1.86,0,0,0,1,0,0,17,Low,High floor
4,flat,G-12,356.0,1,1,1,1,West,Moderately Old,0.51,1,0,0,0,0,0,27,Medium,Low floor


In [5]:
df['price'] =df['price_in_crore']

In [6]:
df['built_up_area']=df['area']

In [7]:
df.sample(7)

,property_type,location,area,bedRoom,bathroom,balcony,floorNum,facing,agePossession,price_in_crore,...,Store Room,Servant Room,Prayer Room,Pooja Room,furnishing_type,luxury_score,luxury_category,floor_category,price,built_up_area
3777,flat,DHA Phase 3,891.0,2,2,2,1,North,Moderately Old,1.61,...,0,0,0,0,1,45,High,Low floor,1.61,891.0
1578,flat,F-8,910.0,3,3,2,5,North,Moderately Old,1.88,...,0,0,0,0,2,18,Low,Mid floor,1.88,910.0
2852,flat,F-7,452.0,1,1,2,8,West,Just Completed,1.39,...,0,0,1,0,0,52,High,High floor,1.39,452.0
1953,flat,F-8,1156.0,3,4,1,1,South-East,Relatively New,2.79,...,0,1,0,0,0,52,High,Low floor,2.79,1156.0
2542,flat,DHA Phase 3,821.0,2,2,2,0,South,New Property,1.10,...,1,0,0,0,1,44,High,Low floor,1.10,821.0
2583,house,G-10,1362.0,6,6,0,5,West,Relatively New,2.30,...,0,0,0,1,0,42,Medium,Mid floor,2.30,1362.0
3899,flat,DHA Phase 1,1170.0,3,3,0,0,South,Old Property,1.61,...,0,0,1,0,1,28,Medium,Low floor,1.61,1170.0


In [8]:
df = df.drop(columns = ['Pooja Room', 'Study Room', 'Prayer Room','price_in_crore','area','facing'])

In [9]:
df.head()

,property_type,location,bedRoom,bathroom,balcony,floorNum,agePossession,Store Room,Servant Room,furnishing_type,luxury_score,luxury_category,floor_category,price,built_up_area
0,flat,DHA Phase 1,2,2,1,3,Moderately Old,0,0,1,17,Low,Mid floor,1.87,793.0
1,flat,G-15,2,2,3,3,New Property,0,0,1,39,Medium,Mid floor,1.87,870.0
2,flat,Bahria Town,3,3,2,6,Under Construction,1,0,2,8,Low,High floor,1.84,947.0
3,flat,G-12,3,4,3,9,Old Property,0,0,0,17,Low,High floor,1.86,1322.0
4,flat,G-12,1,1,1,1,Moderately Old,0,0,0,27,Medium,Low floor,0.51,356.0


In [10]:
df['furnishing_type'].value_counts()

furnishing_type
0    1778
1    1274
2     952
Name: count, dtype: int64

In [11]:
# 0 -> unfurnished
# 1 -> semifurnished
# 2 -> furnished
df['furnishing_type']=df['furnishing_type'].replace({0:'unfurnished',1:'semifurnished',2:'furnished'})

In [12]:
df.head()

,property_type,location,bedRoom,bathroom,balcony,floorNum,agePossession,Store Room,Servant Room,furnishing_type,luxury_score,luxury_category,floor_category,price,built_up_area
0,flat,DHA Phase 1,2,2,1,3,Moderately Old,0,0,semifurnished,17,Low,Mid floor,1.87,793.0
1,flat,G-15,2,2,3,3,New Property,0,0,semifurnished,39,Medium,Mid floor,1.87,870.0
2,flat,Bahria Town,3,3,2,6,Under Construction,1,0,furnished,8,Low,High floor,1.84,947.0
3,flat,G-12,3,4,3,9,Old Property,0,0,unfurnished,17,Low,High floor,1.86,1322.0
4,flat,G-12,1,1,1,1,Moderately Old,0,0,unfurnished,27,Medium,Low floor,0.51,356.0


In [13]:
X = df.drop(columns = ['price'])
Y = df['price']

In [14]:
#apply log1p 
Y_transformed = np.log1p(Y)

In [15]:
columns_to_encode = ['property_type','location','agePossession','balcony','furnishing_type','luxury_category','floor_category']

In [16]:
#creating columntransformer
preprocessor = ColumnTransformer(
    transformers =[
    
        ('num',StandardScaler(),['bedRoom', 'bathroom', 'built_up_area', 'Servant Room', 'Store Room']),
        ('cat',OrdinalEncoder(),columns_to_encode),

    
] ,remainder = 'passthrough')

In [17]:
#creating pipeline
pipeline = Pipeline([
    ('preprocessor',preprocessor),
    ('regressor',LinearRegression())
])

In [18]:
# K-fold cross-validation
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score(pipeline, X, Y_transformed, cv=kfold, scoring='r2')

In [19]:
scores.mean(),scores.std()

(0.5697323957596898, 0.04432329631419345)

In [20]:
X_train,X_test,Y_train,Y_test = train_test_split(X,Y_transformed, test_size = 0.2, random_state = 42)

In [21]:
pipeline.fit(X_train,Y_train)

,steps,"[('preprocessor', ...), ('regressor', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [22]:
Y_pred = pipeline.predict(X_test)

In [23]:
#to reverse the log transformation
Y_pred = np.expm1(Y_pred)

In [24]:
mean_absolute_error(np.expm1(Y_test),Y_pred)

0.6814395949888902

In [25]:
def scorer(model_name, model):
    output = []
    output.append(model_name)
    
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('regressor', model)
    ])
    
    # K-fold cross-validation
    kfold = KFold(n_splits=10, shuffle=True, random_state=42)
    scores = cross_val_score(pipeline, X, Y_transformed, cv=kfold, scoring='r2')
    
    output.append(scores.mean())
    
    X_train, X_test, Y_train, Y_test = train_test_split(X,Y_transformed,test_size=0.2,random_state=42)
    
    pipeline.fit(X_train,Y_train)
    
    Y_pred = pipeline.predict(X_test)
    
    Y_pred = np.expm1(Y_pred)
    
    output.append(mean_absolute_error(np.expm1(Y_test),Y_pred))
    
    return output

In [26]:
pip install xgboost

Note: you may need to restart the kernel to use updated packages.


In [27]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import AdaBoostRegressor
from xgboost import XGBRegressor
from sklearn.neural_network import MLPRegressor

In [28]:
model_dict = {
    'linear_reg':LinearRegression(),
    'svr':SVR(),
    'LASSO':Lasso(),
    'decision tree': DecisionTreeRegressor(),
    'random forest':RandomForestRegressor(),
    'extra trees': ExtraTreesRegressor(),
    'gradient boosting': GradientBoostingRegressor(),
    'adaboost': AdaBoostRegressor(),
    'mlp': MLPRegressor(),
    'xgboost':XGBRegressor()    
}

In [29]:
model_output = []
for model_name,model in model_dict.items():
    model_output.append(scorer(model_name, model))

C:\Users\SAEEDCOMPUTERS\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


In [30]:

model_df = pd.DataFrame(model_output, columns=['name','r2','mae'])

In [31]:
model_df.sort_values(['mae'])

,name,r2,mae
6,gradient boosting,0.883309,0.370851
4,random forest,0.869112,0.380509
9,xgboost,0.868659,0.392252
5,extra trees,0.851879,0.409870
3,decision tree,0.757845,0.503858
7,adaboost,0.759158,0.515215
1,svr,0.653227,0.636235
0,linear_reg,0.569732,0.681440
8,mlp,0.636437,0.715472
2,LASSO,-0.002691,1.004720


In [32]:
#creating a column tansformer for precessing
preprocessor = ColumnTransformer(
    transformers =[
        ('num',StandardScaler(),['bedRoom', 'bathroom', 'built_up_area', 'Servant Room', 'Store Room']),
        ('cat',OrdinalEncoder(),columns_to_encode),
        ('cat1',OneHotEncoder(drop  = 'first' , sparse_output = False),['location','agePossession'])
    ],
    remainder =  'passthrough'
)

In [33]:
pipeline = Pipeline([
    ('preprocessor',preprocessor),
    ('regressor',LinearRegression())
])

In [34]:
# K-fold cross-validation
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score(pipeline, X, Y_transformed, cv=kfold, scoring='r2')

In [35]:
scores.mean()

0.8578024092226053

In [36]:
scores.std()

0.012106644985170492

In [37]:
X_train,X_test,Y_train,Y_test = train_test_split(X,Y_transformed,test_size = 0.2,random_state = 42)

In [38]:
pipeline.fit(X_train,Y_train)

,steps,"[('preprocessor', ...), ('regressor', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...), ...]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [39]:
Y_pred = pipeline.predict(X_test)

In [40]:
Y_pred = np.expm1(Y_pred)

In [41]:
mean_absolute_error(np.expm1(Y_test),Y_pred)

0.4174193838111008

In [42]:
def scorer(model_name, model):                  
    
    output = []
    
    output.append(model_name)
    
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('pca', PCA(n_components=0.95)),
        ('regressor', model)
    ])
    
    # K-fold cross-validation
    kfold = KFold(n_splits=10, shuffle=True, random_state=42)
    scores = cross_val_score(pipeline, X, Y_transformed, cv=kfold, scoring='r2')
    
    output.append(scores.mean())
    
    X_train, X_test, Y_train, Y_test = train_test_split(X,Y_transformed,test_size=0.2,random_state=42)
    
    pipeline.fit(X_train,Y_train)
    
    Y_pred = pipeline.predict(X_test)
    
    Y_pred = np.expm1(Y_pred)
    
    output.append(mean_absolute_error(np.expm1(Y_test),Y_pred))
    
    return output

In [43]:

model_dict = {
    'linear_reg':LinearRegression(),
    'svr':SVR(),
    'LASSO':Lasso(),
    'decision tree': DecisionTreeRegressor(),
    'random forest':RandomForestRegressor(),
    'extra trees': ExtraTreesRegressor(),
    'gradient boosting': GradientBoostingRegressor(),
    'adaboost': AdaBoostRegressor(),
    'mlp': MLPRegressor(),
    'xgboost':XGBRegressor()
}

In [44]:
model_output = []
for model_name,model in model_dict.items():
    model_output.append(scorer(model_name, model))

In [45]:

model_df = pd.DataFrame(model_output, columns=['name','r2','mae'])

In [46]:

model_df.sort_values(['mae'])

,name,r2,mae
9,xgboost,0.671475,0.561486
4,random forest,0.588352,0.644062
6,gradient boosting,0.526760,0.662717
5,extra trees,0.555186,0.686167
3,decision tree,0.256554,0.889014
7,adaboost,0.162063,0.958746
1,svr,0.072617,0.960588
8,mlp,0.034099,0.980784
0,linear_reg,0.022427,0.997044
2,LASSO,-0.002691,1.004720


In [47]:
import category_encoders as ce

columns_to_encode = ['property_type','location', 'balcony', 'agePossession', 'furnishing_type', 'luxury_category', 'floor_category']

# Creating a column transformer for preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['bedRoom', 'bathroom', 'built_up_area', 'Servant Room', 'Store Room']),
        ('cat', OrdinalEncoder(), columns_to_encode),
        ('cat1',OneHotEncoder(drop='first',sparse_output=False),['agePossession']),
        ('target_enc', ce.TargetEncoder(), ['location'])
    ], 
    remainder='passthrough'
)

In [48]:
# Creating a pipeline
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

In [49]:
# K-fold cross-validation
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score(pipeline, X, Y_transformed, cv=kfold, scoring='r2')

In [50]:
scores.mean(),scores.std()

(0.8300003421240094, 0.018611651132735513)

In [51]:
def scorer(model_name, model):
    
    output = []
    
    output.append(model_name)
    
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('regressor', model)
    ])
    
    # K-fold cross-validation
    kfold = KFold(n_splits=10, shuffle=True, random_state=42)
    scores = cross_val_score(pipeline, X, Y_transformed, cv=kfold, scoring='r2')
    
    output.append(scores.mean())
    
    X_train, X_test, Y_train, Y_test = train_test_split(X,Y_transformed,test_size=0.2,random_state=42)
    
    pipeline.fit(X_train,Y_train)
    
    Y_pred = pipeline.predict(X_test)
    
    Y_pred = np.expm1(Y_pred)
    
    output.append(mean_absolute_error(np.expm1(Y_test),Y_pred))
    
    return output

In [52]:
model_dict = {
    'linear_reg':LinearRegression(),
    'svr':SVR(),
    'LASSO':Lasso(),
    'decision tree': DecisionTreeRegressor(),
    'random forest':RandomForestRegressor(),
    'extra trees': ExtraTreesRegressor(),
    'gradient boosting': GradientBoostingRegressor(),
    'adaboost': AdaBoostRegressor(),
    'mlp': MLPRegressor(),
    'xgboost':XGBRegressor()
}

In [53]:
model_output = []
for model_name,model in model_dict.items():
    model_output.append(scorer(model_name, model))

In [54]:
model_df = pd.DataFrame(model_output, columns=['name','r2','mae'])

In [55]:
model_df.sort_values(['mae'])

,name,r2,mae
6,gradient boosting,0.893888,0.351069
4,random forest,0.884124,0.366242
5,extra trees,0.879739,0.378576
7,adaboost,0.866422,0.379724
9,xgboost,0.868404,0.384575
3,decision tree,0.782440,0.453421
8,mlp,0.831028,0.455989
0,linear_reg,0.830000,0.456521
1,svr,0.759893,0.540430
2,LASSO,-0.002691,1.004720


In [56]:
from sklearn.model_selection import GridSearchCV

In [64]:
param_grid = {
    'regressor__n_estimators': [50, 100, 200, 300],
    'regressor__max_depth': [None, 10, 20, 30],
    'regressor__max_samples':[0.1, 0.25, 0.5, 1.0],
    'regressor__max_features': ['auto', 'sqrt',None]
}

In [66]:
columns_to_encode = ['property_type','location', 'balcony', 'agePossession', 'furnishing_type', 'luxury_category', 'floor_category']

# Creating a column transformer for preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['bedRoom', 'bathroom', 'built_up_area', 'Servant Room', 'Store Room']),
        ('cat', OrdinalEncoder(), columns_to_encode),
        ('cat1',OneHotEncoder(drop='first',sparse_output=False),['agePossession']),
        ('target_enc', ce.TargetEncoder(), ['location'])
    ], 
    remainder='passthrough'
)

In [68]:
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor())
])

In [70]:
kfold = KFold(n_splits=10, shuffle=True, random_state=42)

In [72]:
#search = GridSearchCV(pipeline, param_grid, cv=kfold, scoring='r2', n_jobs=1, verbose=4)
# First, fit the GridSearchCV
search = GridSearchCV(pipeline, param_grid, cv=kfold, scoring='r2', n_jobs=-1, verbose=4)
search.fit(X, Y_transformed)  # ← This must be executed first!

# Then you can access best_estimator_


Fitting 10 folds for each of 192 candidates, totalling 1920 fits


C:\Users\SAEEDCOMPUTERS\anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py:516: FitFailedWarning: 
640 fits failed out of a total of 1920.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
401 fits failed with the following error:
Traceback (most recent call last):
  File "C:\Users\SAEEDCOMPUTERS\anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py", line 859, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "C:\Users\SAEEDCOMPUTERS\anaconda3\Lib\site-packages\sklearn\base.py", line 1365, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\SAEEDCOMPUTERS\anaconda3\Lib\site-packages\sklearn\pipeline.py", line 6

,estimator,Pipeline(step...Regressor())])
,param_grid,"{'regressor__max_depth': [None, 10, ...], 'regressor__max_features': ['auto', 'sqrt', ...], 'regressor__max_samples': [0.1, 0.25, ...], 'regressor__n_estimators': [50, 100, ...]}"
,scoring,'r2'
,n_jobs,-1
,refit,True
,cv,KFold(n_split... shuffle=True)
,verbose,4
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,transformers,"[('num', ...), ('cat', ...), ...]"


In [74]:
#search.fit(X, Y_transformed)
final_pipe = search.best_estimator_

In [76]:
final_pipe = search.best_estimator_

In [78]:
search.best_score_

0.8879128915304804

In [82]:

final_pipe.fit(X,Y_transformed)

,steps,"[('preprocessor', ...), ('regressor', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...), ...]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [84]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['bedRoom', 'bathroom', 'built_up_area', 'Servant Room', 'Store Room']),
        ('cat', OrdinalEncoder(), columns_to_encode),
        ('cat1',OneHotEncoder(drop='first',sparse_output=False),['location','agePossession'])
    ], 
    remainder='passthrough'
)

In [86]:
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(n_estimators=500))
])

In [90]:
pipeline.fit(X,Y_transformed)

,steps,"[('preprocessor', ...), ('regressor', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...), ...]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [92]:
import pickle

with open('pipeline.pkl', 'wb') as file:
    pickle.dump(pipeline, file)

In [94]:
with open('df.pkl', 'wb') as file:
    pickle.dump(X, file)

In [96]:
X

,property_type,location,bedRoom,bathroom,balcony,floorNum,agePossession,Store Room,Servant Room,furnishing_type,luxury_score,luxury_category,floor_category,built_up_area
0,flat,DHA Phase 1,2,2,1,3,Moderately Old,0,0,semifurnished,17,Low,Mid floor,793.0
1,flat,G-15,2,2,3,3,New Property,0,0,semifurnished,39,Medium,Mid floor,870.0
2,flat,Bahria Town,3,3,2,6,Under Construction,1,0,furnished,8,Low,High floor,947.0
3,flat,G-12,3,4,3,9,Old Property,0,0,unfurnished,17,Low,High floor,1322.0
4,flat,G-12,1,1,1,1,Moderately Old,0,0,unfurnished,27,Medium,Low floor,356.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3999,flat,DHA Phase 1,3,3,1,0,Just Completed,1,0,furnished,33,Medium,Low floor,935.0
4000,flat,F-7,3,3,0,7,New Property,0,1,semifurnished,34,Medium,High floor,1196.0
4001,house,G-10,16,17,0,4,Relatively New,0,0,unfurnished,33,Medium,Mid floor,3627.0
4002,flat,G-15,4,4,3,4,Relatively New,0,1,furnished,45,High,Mid floor,1480.0


In [98]:
X.columns

Index(['property_type', 'location', 'bedRoom', 'bathroom', 'balcony',
       'floorNum', 'agePossession', 'Store Room', 'Servant Room',
       'furnishing_type', 'luxury_score', 'luxury_category', 'floor_category',
       'built_up_area'],
      dtype='object')

In [373]:
data = [['house', 'G-11', 4, 3, 3, 'New Property',  0, 0, 'unfurnished', 'Low', 'Low floor',2750,]]
columns = ['property_type', 'location', 'bedRoom', 'bathroom', 'balcony',
       'agePossession', 'Store Room', 'Servant Room',
       'furnishing_type', 'luxury_category', 'floor_category', 'built_up_area',]

# Convert to DataFrame
one_df = pd.DataFrame(data, columns=columns)

one_df

,property_type,location,bedRoom,bathroom,balcony,agePossession,Store Room,Servant Room,furnishing_type,luxury_category,floor_category,built_up_area
0,house,G-11,4,3,3,New Property,0,0,unfurnished,Low,Low floor,2750


In [375]:
np.expm1(pipeline.predict(one_df))

array([4.6890542])